# Importações

In [ ]:
!pip install tensorflow #usar redes neurais pré-treinadas

import os #manipulação de diretórios e caminhos de arquivos
import pandas as pd
import numpy as np
from tensorflow.keras.applications import MobileNetV2#pré-processamento específico para MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input#pré-processamento específico para MobileNetV2
from tensorflow.keras.preprocessing.image import load_img, img_to_array# leitura e conversão de imagens para arrays compatíveis com Keras
from tqdm import tqdm#criação de barras de progresso durante loops longos

# Carregamento de dados

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

caminho_treino = '/content/drive/MyDrive/visão computacional/treino'
caminho_teste = '/content/drive/MyDrive/visão computacional/teste'


Mounted at /content/drive


# Vetorização com Embbedings

* Modelo Base = Modelo pré treinado para embeddings


In [ ]:
modelo_base = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")

/tmp/ipython-input-3-605486222.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  modelo_base = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
def gerar_embedding(caminho_imagem):
    imagem = load_img(caminho_imagem, target_size=(224, 224))
    imagem = img_to_array(imagem)
    imagem = np.expand_dims(imagem, axis=0)
    imagem = preprocess_input(imagem)
    embedding = modelo_base.predict(imagem, verbose=0)
    return embedding.flatten()


In [ ]:
def gerar_dataset_com_embeddings(diretorio_base):
    dados = []
    labels = []
    classes = os.listdir(diretorio_base)
    for classe in classes:
        pasta = os.path.join(diretorio_base, classe)
        if not os.path.isdir(pasta):
            continue
        for arquivo in tqdm(os.listdir(pasta), desc=f"Processando {classe}"):
            if arquivo.lower().endswith(('jpg', 'jpeg', 'png')):
                caminho = os.path.join(pasta, arquivo)
                embedding = gerar_embedding(caminho)
                dados.append(embedding)
                labels.append(classe)
    df = pd.DataFrame(dados)
    df['label'] = labels
    return df


* Chamada de Função para vetorizar o treino e teste

In [ ]:
df_treino = gerar_dataset_com_embeddings(caminho_treino)
df_teste = gerar_dataset_com_embeddings(caminho_teste)

# Salvando os datasets

caminho_saida = "/content/drive/MyDrive/visão computacional/"
df_treino.to_csv(os.path.join(caminho_saida, 'embeddings_treino.csv'), index=False)
df_teste.to_csv(os.path.join(caminho_saida, 'embeddings_teste.csv'), index=False)


Processando gatos: 100%|██████████| 1012/1012 [04:04<00:00,  4.13it/s]


# Separação de Atributos e Rotulos

In [ ]:
df_treino = pd.read_csv('/content/drive/MyDrive/visão computacional/embeddings_treino.csv')
df_teste = pd.read_csv('/content/drive/MyDrive/visão computacional/embeddings_teste.csv')

In [ ]:
X_treino = df_treino.drop('label', axis=1).values #
y_treino = df_treino['label'].values

X_teste = df_teste.drop('label', axis=1).values
y_teste = df_teste['label'].values


# Normalização

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_treino_normalizado = scaler.fit_transform(X_treino)#calcula minimo e maximo/aplica a transformação nos dados
X_teste_normalizado = scaler.transform(X_teste) #aplica a transformação usando os parâmetros já calculados no fit


# Treinamento

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

modelo_mlp = MLPClassifier(random_state=42, max_iter=300)
modelo_mlp.fit(X_treino_normalizado, y_treino)

y_predito = modelo_mlp.predict(X_teste_normalizado)
print(classification_report(y_teste, y_predito))


              precision    recall  f1-score   support

   cachorros       0.99      0.99      0.99      1012
       gatos       0.99      0.99      0.99      1011

    accuracy                           0.99      2023
   macro avg       0.99      0.99      0.99      2023
weighted avg       0.99      0.99      0.99      2023



# Teste Pratico

In [ ]:
# usand o mesmo modelo base de antes
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def gerar_embedding_imagem_unica(caminho_imagem):
    imagem = load_img(caminho_imagem, target_size=(224, 224))
    imagem = img_to_array(imagem)
    imagem = np.expand_dims(imagem, axis=0)
    imagem = preprocess_input(imagem)
    embedding = modelo_base.predict(imagem, verbose=0)
    return embedding.flatten()


In [ ]:
import os

caminho_novas_imagens = '/content/drive/MyDrive/visão computacional/teste pratico'

for arquivo in os.listdir(caminho_novas_imagens):
    if arquivo.lower().endswith(('.jpg', '.jpeg', '.png')):
        caminho_imagem = os.path.join(caminho_novas_imagens, arquivo)
        vetor_embedding = gerar_embedding_imagem_unica(caminho_imagem)
        vetor_embedding_normalizado = scaler.transform([vetor_embedding])  # Normalizando com o mesmo scaler de antes

        predicao = modelo_mlp.predict(vetor_embedding_normalizado)
        print(f'Imagem: {arquivo} → Predição: {predicao[0]}')


Imagem: download.jpg → Predição: gatos
Imagem: download (1).jpg → Predição: gatos
Imagem: download (2).jpg → Predição: cachorros
Imagem: download (3).jpg → Predição: gatos
Imagem: download (4).jpg → Predição: gatos
Imagem: download (5).jpg → Predição: cachorros
Imagem: download (6).jpg → Predição: cachorros
Imagem: download (7).jpg → Predição: cachorros
Imagem: images.jpg → Predição: cachorros
